# 07 · Layer-choice control: is it layer 18, or just a small adapter?

The restricted adapter (one matrix in layer 18) matched the broad adapter's safety with half its
over-refusal. This notebook tests whether **the layer** matters: it trains the *same* one-matrix LoRA
(`layers.N.mlp.down_proj`, 208,896 parameters for every N) on layer 18 (chosen by activation analysis),
layer 15 (runner-up) and layers 9 and 30 (never in the top 5), with two seeds each, and evaluates
every arm exactly like notebook 05. Plan and decision rule: [`docs/LAYER_CONTROL_PLAN.md`](../docs/LAYER_CONTROL_PLAN.md).

**Needs a G4, L4 or A100 GPU** (about 75–90 minutes for 8 arms on a G4; a T4 is far too slow) and a
Hugging Face token with access to Tiny Aya Water (Colab secret `HF_TOKEN`). The run is resumable:
finished arms are skipped if you rerun after a disconnect. Results of our run are already in
`results/layer_control/`; to only look at them, skip to section 3.

## 1. Setup

In [ ]:
# Project setup (Colab or local): fetch this repository's code and small data files, then load
# the shared config. Large files come from the project's Hugging Face datasets on first use.
import subprocess
import sys
from pathlib import Path

if Path("/content").exists():                     # Google Colab
    REPO = Path("/content/tiny-aya-safety")
    if not REPO.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/ishane0620/tiny-aya-safety", str(REPO)], check=True)
else:                                             # local checkout: this notebook is in <repo>/notebooks
    REPO = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO / "src"))
from tiny_aya_safety.config import CONFIG, adapter_source, artifact, data_path, output_dir

OUT = output_dir()   # everything this notebook writes goes here (override with TAS_OUTPUT_DIR)
print("repo:", REPO, "| outputs:", OUT)

In [ ]:
import os
if Path("/content").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate", "peft",
                    "torchao", "datasets", "safetune==0.1.16"], check=True)
    if not os.environ.get("HF_TOKEN"):
        try:
            from google.colab import userdata
            os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        except Exception:
            print("No HF_TOKEN secret found; the gated base model download will fail.")

## 2. Train and evaluate every arm (GPU)

Set `LAYERS`/`SEEDS` to run a subset. Output streams below; each arm takes ~10 minutes on a G4.

In [ ]:
LAYERS, SEEDS = [18, 15, 9, 30], [0, 1]
RUN = False   # set True to train; False just analyses the published results in results/layer_control/

if RUN:
    cmd = [sys.executable, str(REPO / "scripts/run_layer_control.py"), "--layers", *map(str, LAYERS),
           "--seeds", *map(str, SEEDS), "--out", str(OUT / "layer_control")]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
        for line in p.stdout:
            print(line, end="")
    assert p.returncode == 0

## 3. Results

In [ ]:
answers_dir = (OUT / "layer_control") if RUN else (REPO / "results/layer_control")
subprocess.run([sys.executable, str(REPO / "scripts/analyze_layer_control.py"), str(answers_dir)], check=True)
from IPython.display import Image, Markdown, display
display(Markdown((REPO / "results/layer_control/LAYER_CONTROL_TABLES.md").read_text()))
display(Image(str(REPO / "results/layer_control/layer_control.png")))